In [ ]:

import math
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# =====================================================
# 1. LOAD DATASET
# =====================================================
data = load_breast_cancer()

X = data.data
y = data.target
feature_names = data.feature_names

n_features = X.shape[1]

print("CUCKOO SEARCH FOR DATA MINING")
print("=" * 45)
print("Dataset: Breast Cancer Wisconsin")
print("Total samples:", X.shape[0])
print("Total features:", n_features)
print("Classes:", len(np.unique(y)))


# =====================================================
# 2. SPLIT DATA INTO TRAINING AND TEST SETS
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Validation set is used for feature selection only.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)


# =====================================================
# 3. CREATE SVM CLASSIFIER
# =====================================================
def create_model():
    return make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=1.0, gamma="scale")
    )


# =====================================================
# 4. FITNESS FUNCTION
# Lower fitness is better.
# =====================================================
ALPHA = 0.99

def fitness(mask):
    # Reject an empty feature subset.
    if not np.any(mask):
        return float("inf")

    model = create_model()

    model.fit(X_fit[:, mask], y_fit)
    predictions = model.predict(X_val[:, mask])

    accuracy = accuracy_score(y_val, predictions)
    selected_ratio = np.sum(mask) / n_features

    score = (
        ALPHA * (1.0 - accuracy)
        + (1.0 - ALPHA) * selected_ratio
    )

    return score


# =====================================================
# 5. LEVY FLIGHT USING MANTEGNA'S METHOD
# =====================================================
def levy_flight(size, beta=1.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    numerator = (
        math.gamma(1 + beta)
        * math.sin(math.pi * beta / 2)
    )

    denominator = (
        math.gamma((1 + beta) / 2)
        * beta
        * (2 ** ((beta - 1) / 2))
    )

    sigma_u = (numerator / denominator) ** (1 / beta)

    u = rng.normal(0, sigma_u, size=size)
    v = rng.normal(0, 1, size=size)

    step = u / (np.abs(v) ** (1 / beta) + 1e-12)

    return step


# =====================================================
# 6. CONVERT CONTINUOUS POSITIONS TO BINARY FEATURES
# =====================================================
def to_binary(position, rng):
    position = np.clip(position, -20, 20)

    probabilities = 1.0 / (1.0 + np.exp(-position))

    mask = rng.random(len(position)) < probabilities

    # At least one feature must be selected.
    if not np.any(mask):
        mask[np.argmax(position)] = True

    return mask


# =====================================================
# 7. CUCKOO SEARCH ALGORITHM
# =====================================================
def cuckoo_search(
    n_nests=15,
    n_iterations=25,
    discovery_probability=0.25,
    step_size=0.15,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    # Initialize random nests.
    nests = rng.integers(
        0, 2,
        size=(n_nests, n_features)
    ).astype(bool)

    # Avoid empty nests.
    for nest in nests:
        if not np.any(nest):
            nest[rng.integers(n_features)] = True

    # Evaluate initial fitness.
    scores = np.array([
        fitness(nest) for nest in nests
    ])

    best_index = np.argmin(scores)
    best_nest = nests[best_index].copy()
    best_score = scores[best_index]

    history = [best_score]

    # Main optimization loop.
    for iteration in range(n_iterations):

        # Generate new solutions using Levy flights.
        for i in range(n_nests):

            levy = levy_flight(
                n_features,
                beta=1.5,
                rng=rng
            )

            position = (
                nests[i].astype(float)
                + step_size * levy
            )

            candidate = to_binary(position, rng)
            candidate_score = fitness(candidate)

            # Choose a random nest to compete with.
            j = rng.integers(n_nests)

            if candidate_score < scores[j]:
                nests[j] = candidate
                scores[j] = candidate_score

        # Abandon the worst nests.
        n_abandon = max(
            1,
            int(discovery_probability * n_nests)
        )

        worst_indices = np.argsort(scores)[-n_abandon:]

        for j in worst_indices:

            candidate = rng.integers(
                0, 2,
                size=n_features
            ).astype(bool)

            if not np.any(candidate):
                candidate[rng.integers(n_features)] = True

            candidate_score = fitness(candidate)

            nests[j] = candidate
            scores[j] = candidate_score

        # Update global best solution.
        current_index = np.argmin(scores)

        if scores[current_index] < best_score:
            best_nest = nests[current_index].copy()
            best_score = scores[current_index]

        history.append(best_score)

        print(
            f"Iteration {iteration + 1:02d}/{n_iterations} | "
            f"Best fitness: {best_score:.5f} | "
            f"Selected features: {np.sum(best_nest)}"
        )

    return best_nest, best_score, history


# =====================================================
# 8. RUN CUCKOO SEARCH
# =====================================================
print("\nStarting Cuckoo Search feature selection...\n")

best_mask, best_fitness, history = cuckoo_search()

selected_features = feature_names[best_mask]

print("\n" + "=" * 45)
print("BEST FEATURE SUBSET")
print("=" * 45)

for feature in selected_features:
    print("-", feature)

selected_count = int(np.sum(best_mask))

print("\nSelected features:", selected_count, "/", n_features)
print("Best validation fitness:", round(best_fitness, 5))


# =====================================================
# 9. TRAIN BASELINE SVM USING ALL FEATURES
# =====================================================
baseline_model = create_model()

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)


# =====================================================
# 10. TRAIN OPTIMIZED SVM USING SELECTED FEATURES
# =====================================================
optimized_model = create_model()

optimized_model.fit(
    X_train[:, best_mask],
    y_train
)

optimized_predictions = optimized_model.predict(
    X_test[:, best_mask]
)


# =====================================================
# 11. EVALUATION FUNCTION
# =====================================================
def evaluate_model(title, y_true, y_pred, count):

    print("\n" + title)
    print("-" * 45)

    print(f"Features used : {count}/{n_features}")

    print(
        f"Accuracy      : "
        f"{accuracy_score(y_true, y_pred):.4f}"
    )

    print(
        f"Precision     : "
        f"{precision_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"Recall        : "
        f"{recall_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"F1-score      : "
        f"{f1_score(y_true, y_pred, zero_division=0):.4f}"
    )


# =====================================================
# 12. DISPLAY FINAL RESULTS
# =====================================================
evaluate_model(
    "BASELINE SVM - ALL FEATURES",
    y_test,
    baseline_predictions,
    n_features
)

evaluate_model(
    "CUCKOO SEARCH + SVM",
    y_test,
    optimized_predictions,
    selected_count
)


# =====================================================
# 13. FEATURE REDUCTION
# =====================================================
feature_reduction = (
    (n_features - selected_count) / n_features
) * 100

print("\n" + "=" * 45)
print("FINAL SUMMARY")
print("=" * 45)

print("Original features :", n_features)
print("Selected features :", selected_count)
print(f"Feature reduction : {feature_reduction:.2f}%")
print(f"Best fitness      : {best_fitness:.5f}")

print("\nExecution completed successfully.")


# VARALAKSMI SR [1BM2CS537]


CUCKOO SEARCH FOR DATA MINING
Dataset: Breast Cancer Wisconsin
Total samples: 569
Total features: 30
Classes: 2

Starting Cuckoo Search feature selection...

Iteration 01/25 | Best fitness: 0.02204 | Selected features: 14
Iteration 02/25 | Best fitness: 0.02204 | Selected features: 14
Iteration 03/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 04/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 05/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 06/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 07/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 08/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 09/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 10/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 11/25 | Best fitness: 0.01502 | Selected features: 19
Iteration 12/25 | Best fitness: 0.01368 | Selected features: 15
Iteration 13/25 | Best fitness: 0.01368 | Selected features: 15
Iteration 